In [ ]:
import sys,os
sys.path.append('../../package/')
%run ../../package/init_jupyter.py

In [ ]:
import torch
# from models import load_pretrained
# from loader import get_dataloader
# from geometry import get_pullbacked_Riemannian_metric
from mpl_toolkits.mplot3d import Axes3D  # 3D plotting module
import mujoco as mj
device = f'cuda:0'
robot_chars = RobotCharacteristics()

### drawing kinematic structure tree of robots
I do not recommend to run this line except for loss of tree pictures or adding more robots in this dataset

In [ ]:
draw_trees = False
plot_trees = False

In [ ]:
if draw_trees:
    robot_chars._draw_trees(verbose=plot_trees)

# Data Curation

## Data before curation

In [ ]:
vis_robots=False

In [ ]:
if vis_robots:
    robot_chars.curate=False
    robot_chars._initialize_robots()
    for i in range(len(robot_chars.robot_list)):
        robot_name=robot_chars.robot_list[i]
        robot_chars._initialize_env(robot_name)
        robot_chars.visualize_robot_joints_with_arrows(robot_name)

## Data after curation

In [ ]:
vis_curated_robots = True

In [ ]:
if vis_curated_robots:
    robot_chars.curate=True
    robot_chars._initialize_robots()
    curated_robots = ['tiago_dual','op3','icub','jaxon','poppy','armar4','robonaut2']
    for i in range(len(curated_robots)):
        robot_name=curated_robots[i]
        robot_chars._initialize_env(robot_name)
        robot_chars.visualize_robot_joints_with_arrows(robot_name)

# Preprocessing

In [ ]:
preprocess=True
visualize_preprocessed_data=True


TODO: the virtual wrist is not reflected in the classifying_joint_info used by visualize_data_to_classify; where does it go wrong?

In [ ]:
if preprocess:
    robot_chars.curate=True

    robot_chars._initialize_robots()
    dataset = robot_chars.create_data_to_classify(save=False,repr_kind = "dh")
    dataset2 = robot_chars.create_data_to_classify(save=False, repr_kind = "screw")
    # data_to_visualize = {'op3':robot_chars.classifying_joint_info['op3']['data_to_classify']}
    data_to_visualize = {
        robot_name: info['data_to_classify_dh'] 
        for robot_name, info in robot_chars.classifying_joint_info.items()
        if 'data_to_classify_dh' in info
    }
    
    if visualize_preprocessed_data:
        clip = robot_chars.visualize_data_to_classify(data_to_visualize,
                                                      vis=visualize_preprocessed_data, 
                                                      test=False,# to see robots individually, change this variable to True
                                                      repr_kind="dh"
                                                      )

In [ ]:
robot_chars.classifying_joint_info['talos']['data_to_classify_dh']

In [ ]:
robot_chars.classifying_joint_info['talos']['data_to_classify']

In [ ]:
clip.ipython_display()

In [ ]:
data=np.load('../../data/mjcf_test/data.npy')
label_data = np.load('../../data/mjcf_test/label.npy')
njnt_data = np.load('../../data/mjcf_test/njnt.npy')
indices_data = np.load('../../data/mjcf_test/indices.npy')

robot_chars.max_joints_per_group=njnt_data

njnt = torch.from_numpy(njnt_data).int()
qpos_plus = np.ones(njnt.sum().item()) * (np.pi / 6)   # +30 deg
qpos_minus = np.ones(njnt.sum().item()) * (-np.pi / 6) # -30 deg

qpos_set = torch.tensor(np.stack([qpos_plus, qpos_minus],axis=0), dtype=torch.float64)  # shape (2, 30)
active_joint_indices=torch.from_numpy(indices_data).int()

In [ ]:
np.linalg.norm(data[:,:,:3],axis=-1)
# data[:,:,:3]